# Inference for Raman Spectroscopy

In [ ]:
# imports

In [47]:
import os
import re
import glob
import pickle
import numpy as np
import pandas as pd

In [56]:
import sys
print(sys.executable)

/opt/bioinf/jupyterlab/4.2.5/jupyterlab_4.2.5/bin/python3


In [58]:
import sys
print(sys.version)

3.10.14 | packaged by conda-forge | (main, Mar 20 2024, 12:45:18) [GCC 12.3.0]


In [3]:
!pip list

Package                      Version
---------------------------- --------------
absl-py                      2.1.0
alembic                      1.13.2
annotated-types              0.7.0
anyio                        4.4.0
argon2-cffi                  23.1.0
argon2-cffi-bindings         21.2.0
arrow                        1.3.0
asn1crypto                   1.5.1
astroid                      3.3.8
asttokens                    2.4.1
astunparse                   1.6.3
async_generator              1.10
async-lru                    2.0.4
attrs                        24.2.0
autopep8                     1.5.6
Babel                        2.14.0
bash_kernel                  0.9.3
beautifulsoup4               4.12.3
biopython                    1.84
bleach                       6.1.0
blinker                      1.8.2
bokeh                        3.4.3
branca                       0.7.2
Brotli                       1.1.0
cached-property              1.5.2
certifi                      2024.8.30
c

In [48]:
import zipfile
with zipfile.ZipFile("./Inference_data/24062026.zip", "r") as zip_ref:
    zip_ref.extractall("./Inference_data/")

In [49]:
path_to_data = "./Inference_data/24062026/"

In [50]:
path_to_model = "./Inference_data/24062026/"

### Read spectra exactly like training

In [51]:

def extract_id(filename):
    """
    Extract sample ID from filename.
    Examples:
        23002_1.csv -> 23002
        23007_3.csv -> 23007
    """
    return os.path.splitext(filename)[0]


def read_raman_csv(filepath):

    df = pd.read_csv(filepath, header=None)

    if df.shape[1] < 2:
        raise ValueError(f"Invalid file format: {filepath}")

    # ONLY intensity column
    intensity = df.iloc[:, 1].values

    return intensity


def process_folder(folder_path):

    files = sorted(glob.glob(os.path.join(folder_path, "*.csv")))

    temp = []
    max_len = 0

    # First pass
    for f in files:

        filename = os.path.basename(f)

        sample_id = extract_id(filename)

        intensity = read_raman_csv(f)

        max_len = max(max_len, len(intensity))

        temp.append(
            (
                sample_id,
                filename,
                intensity
            )
        )

    print(f"Longest spectrum length = {max_len}")

    # Second pass
    rows = []

    for sample_id, filename, intensity in temp:

        padded = np.full(max_len, np.nan)

        padded[:len(intensity)] = intensity

        row = [sample_id, filename] + padded.tolist()

        rows.append(row)

    columns = (
        ["id", "filename"]
        + [f"f{i+1}" for i in range(max_len)]
    )

    df = pd.DataFrame(rows, columns=columns)
   
    return df

     
df = process_folder(path_to_data)

print("\nShape:")
print(df.shape)

print("\nFirst 5 rows:")
print(df.head())

print("\nFirst 10 columns:")
print(df.iloc[:, :10])

# In Jupyter Notebook
display(df.head())


Longest spectrum length = 1901

Shape:
(58, 1903)

First 5 rows:
        id     filename        f1        f2        f3        f4        f5  \
0  23002_1  23002_1.csv  668.4283  633.5789  607.0146  585.5553  566.0465   
1  23002_2  23002_2.csv  633.0762  601.9022  573.9601  549.2736  527.8670   
2  23002_3  23002_3.csv  572.9999  550.3541  535.7136  524.2740  511.2693   
3  23003_1  23003_1.csv  628.6609  589.3753  556.8787  532.5725  517.8469   
4  23003_2  23003_2.csv  633.1290  607.3353  590.2379  577.6854  565.5599   

         f6        f7        f8  ...     f1892     f1893     f1894     f1895  \
0  546.0148  526.3439  508.9689  ...  223.4026  219.8773  210.6685  188.7631   
1  509.7803  495.1381  484.0909  ...  228.7232  225.3269  217.5115  193.1440   
2  493.2162  472.9890  455.4468  ...  196.4703  183.8231  178.6757  186.5249   
3  513.3089  515.6324  520.2725  ...  172.4553  174.3590  175.8154  164.2900   
4  550.6349  534.0774  518.4304  ...  210.6297  194.4471  176.7407  185.

,id,filename,f1,f2,f3,f4,f5,f6,f7,f8,...,f1892,f1893,f1894,f1895,f1896,f1897,f1898,f1899,f1900,f1901
0,23002_1,23002_1.csv,668.4283,633.5789,607.0146,585.5553,566.0465,546.0148,526.3439,508.9689,...,223.4026,219.8773,210.6685,188.7631,163.4800,147.2062,144.0273,153.5987,164.1683,157.5827
1,23002_2,23002_2.csv,633.0762,601.9022,573.9601,549.2736,527.8670,509.7803,495.1381,484.0909,...,228.7232,225.3269,217.5115,193.1440,166.1647,154.9333,161.3898,179.8977,195.7359,189.6899
2,23002_3,23002_3.csv,572.9999,550.3541,535.7136,524.2740,511.2693,493.2162,472.9890,455.4468,...,196.4703,183.8231,178.6757,186.5249,186.1582,156.7213,127.0447,140.9624,179.9842,188.7510
3,23003_1,23003_1.csv,628.6609,589.3753,556.8787,532.5725,517.8469,513.3089,515.6324,520.2725,...,172.4553,174.3590,175.8154,164.2900,145.5787,131.1888,138.6051,183.5119,233.2957,228.1483
4,23003_2,23003_2.csv,633.1290,607.3353,590.2379,577.6854,565.5599,550.6349,534.0774,518.4304,...,210.6297,194.4471,176.7407,185.1416,206.3026,218.1592,212.3726,187.5024,156.2827,139.5590


### Preprocessing same as the model training

In [52]:
def preprocess_for_model(df):

    feature_cols = [c for c in df.columns if c.startswith("f")]

    X = (
        df[feature_cols]
        .apply(pd.to_numeric, errors="coerce")
        .values
    )

    X = np.nan_to_num(X, nan=0.0).astype(np.float32)

    # Same normalization used during training
    max_vals = np.max(X, axis=1, keepdims=True)

    max_vals[max_vals == 0] = 1

    X = X / max_vals

    return X


### Load model

In [53]:
def load_model(pickle_file):

    with open(pickle_file, "rb") as f:
        bundle = pickle.load(f)

    model = bundle["model"]
    scaler = bundle["scaler"]

    return model, scaler


### Predict

In [54]:

def predict_folder(folder_path, pickle_file):

    print(f"\nLoading model: {pickle_file}")

    df = prepare_folder(folder_path)

    X = preprocess_for_model(df)

    model, scaler = load_model(pickle_file)

    # SVM has scaler
    if scaler is not None:
        X = scaler.transform(X)

    predictions = model.predict(X)

    probabilities = None

    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)[:, 1]

    results = pd.DataFrame({
        "id": df["id"],
        "filename": df["filename"],
        "prediction": predictions
    })

    if probabilities is not None:
        results["paracetamol_probability"] = probabilities

    # Convert numeric prediction to label
    results["predicted_class"] = results["prediction"].map({
        1: "Paracetamol",
        0: "Not Paracetamol"
    })

    return results

### Run

In [55]:
# path_to_data = "./Inference_data/24062026/"

# -------- SVM --------
svm_results = predict_folder(
    path_to_data,
    "../SVM.pkl"
)

print("\nSVM Predictions")
print(svm_results)

svm_results.to_csv(
    "svm_predictions.csv",
    index=False
)

# -------- XGBoost --------
xgb_results = predict_folder(
    path_to_data,
    "../XGBoost.pkl"
)

print("\nXGBoost Predictions")
print(xgb_results)

xgb_results.to_csv(
    "xgb_predictions.csv",
    index=False
)


Loading model: ../SVM.pkl
Longest spectrum length = 1901

SVM Predictions
         id     filename  prediction  paracetamol_probability  predicted_class
0   23002_1  23002_1.csv           1                 0.448975      Paracetamol
1   23002_2  23002_2.csv           1                 0.445731      Paracetamol
2   23002_3  23002_3.csv           0                 0.442853  Not Paracetamol
3   23003_1  23003_1.csv           1                 0.447917      Paracetamol
4   23003_2  23003_2.csv           0                 0.440923  Not Paracetamol
5   23003_3  23003_3.csv           0                 0.428612  Not Paracetamol
6   23004_1  23004_1.csv           0                 0.428553  Not Paracetamol
7   23004_2  23004_2.csv           0                 0.430371  Not Paracetamol
8   23004_3  23004_3.csv           0                 0.424770  Not Paracetamol
9   23005_1  23005_1.csv           0                 0.431025  Not Paracetamol
10  23005_2  23005_2.csv           0                 0.4

In [43]:
print(os.path.exists("../SVM.pkl"))

True
